# Predição de Notas de Estudantes - Aprendizado Supervisionado
**Curso:** Ciência da Computação (Ano Letivo 2026, 1º Semestre, 7º Período)  
**Disciplina:** Inteligência Artificial II  
**Professor:** Gustavo H. R. Magalhães  

Este notebook apresenta a solução completa para o trabalho de predição da nota final de estudantes (`G3`) utilizando o conjunto de dados **Student Performance** da UCI.  
Para demonstrar o funcionamento dos algoritmos de inteligência artificial de forma didática e profunda, todas as etapas de modelagem (Regressão Linear, Árvore de Decisão e Random Forest) foram desenvolvidas **do zero (scratch)**, sem o uso de bibliotecas de aprendizado prontas como o `scikit-learn`.

## 1. Configuração do Ambiente e Download dos Dados

Primeiro, vamos baixar e descompactar os dados diretamente do repositório da UCI.

In [4]:
# Baixar o dataset
import os
import zipfile
import requests

url = "https://archive.ics.uci.edu/static/public/320/student+performance.zip"
zip_path = "student_performance.zip"
extract_dir = "data"

if not os.path.exists(extract_dir):
    print("Baixando arquivo zip...")
    response = requests.get(url, stream=True)
    with open(zip_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            
    print("Extraindo...")
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
        
    inner_zip = os.path.join(extract_dir, "student.zip")
    if os.path.exists(inner_zip):
        with zipfile.ZipFile(inner_zip, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)
        os.remove(inner_zip)
    os.remove(zip_path)
    print("Download e extração concluídos!")
else:
    print("Diretório de dados já existe.")

Diretório de dados já existe.


## 2. Implementação das Classes do Zero (Scratch)

Abaixo definimos o pré-processador de dados (escalador numérico, codificador categórico One-Hot), os algoritmos de regressão e as funções para cálculo das métricas de avaliação.

In [5]:
import csv
import random
import math

random.seed(42)

def train_test_split(X, y, test_size=0.3, random_state=42):
    """Divide os dados em treino e teste"""
    random.seed(random_state)
    n = len(X)
    indices = list(range(n))
    random.shuffle(indices)
    
    test_count = int(n * test_size)
    test_indices = indices[:test_count]
    train_indices = indices[test_count:]
    
    X_train = [X[i] for i in train_indices]
    X_test = [X[i] for i in test_indices]
    y_train = [y[i] for i in train_indices]
    y_test = [y[i] for i in test_indices]
    
    return X_train, X_test, y_train, y_test

class StandardScalerScratch:
    """Normalizador de dados (Z-Score)"""
    def __init__(self):
        self.means = []
        self.stds = []

    def fit(self, X):
        n_features = len(X[0])
        n_samples = len(X)
        self.means = []
        self.stds = []
        for j in range(n_features):
            col = [X[i][j] for i in range(n_samples)]
            mean = sum(col) / n_samples
            variance = sum((x - mean) ** 2 for x in col) / n_samples
            std = math.sqrt(variance) if variance > 0 else 1.0
            self.means.append(mean)
            self.stds.append(std)
            
    def transform(self, X):
        return [[(row[j] - self.means[j]) / self.stds[j] for j in range(len(row))] for row in X]

class OneHotEncoderScratch:
    """Codificador de variáveis categóricas"""
    def __init__(self):
        self.categories_map = {}
        self.cat_cols = []
        self.num_cols = []
        self.feature_names = []

    def fit(self, data, cat_cols, num_cols):
        self.cat_cols = cat_cols
        self.num_cols = num_cols
        for col in cat_cols:
            unique_vals = sorted(list(set(row[col] for row in data)))
            if len(unique_vals) > 1:
                self.categories_map[col] = unique_vals[1:]
            else:
                self.categories_map[col] = unique_vals
        
        self.feature_names = list(num_cols)
        for col in cat_cols:
            for val in self.categories_map[col]:
                self.feature_names.append(f"{col}_{val}")

    def transform(self, data):
        X = []
        for row in data:
            new_row = [float(row[col]) for col in self.num_cols]
            for col in self.cat_cols:
                val = row[col]
                for cat in self.categories_map[col]:
                    new_row.append(1.0 if val == cat else 0.0)
            X.append(new_row)
        return X

# ==================== MODELOS ====================

class LinearRegressionGD:
    """Regressão Linear via Gradiente Descendente"""
    def __init__(self, lr=0.01, epochs=1000):
        self.lr = lr
        self.epochs = epochs
        self.weights = []
        self.bias = 0.0

    def fit(self, X, y):
        n_samples = len(X)
        n_features = len(X[0])
        self.weights = [0.0] * n_features
        self.bias = 0.0
        
        for epoch in range(self.epochs):
            dw = [0.0] * n_features
            db = 0.0
            for i in range(n_samples):
                y_pred = sum(X[i][j] * self.weights[j] for j in range(n_features)) + self.bias
                error = y_pred - y[i]
                for j in range(n_features):
                    dw[j] += error * X[i][j]
                db += error
            for j in range(n_features):
                self.weights[j] -= (self.lr * 2.0 / n_samples) * dw[j]
            self.bias -= (self.lr * 2.0 / n_samples) * db
            
    def predict(self, X):
        return [sum(row[j] * self.weights[j] for j in range(len(self.weights))) + self.bias for row in X]

class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value
    def is_leaf(self): return self.value is not None

class DecisionTreeRegressorScratch:
    """Árvore de Regressão"""
    def __init__(self, max_depth=5, min_samples_split=5, max_features=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.root = None

    def fit(self, X, y):
        self.root = self._build_tree(X, y, depth=0)

    def _calculate_sse(self, y):
        if not y: return 0.0
        mean = sum(y) / len(y)
        return sum((val - mean) ** 2 for val in y)

    def _best_split(self, X, y, feature_indices):
        best_sse = float('inf')
        best_idx, best_thresh = None, None
        n_samples = len(X)
        if n_samples < self.min_samples_split: return None, None
        
        for idx in feature_indices:
            values = [row[idx] for row in X]
            unique_vals = sorted(list(set(values)))
            if len(unique_vals) <= 1: continue
            
            thresholds = []
            if len(unique_vals) > 10:
                step = len(unique_vals) // 10
                thresholds = [unique_vals[k * step] for k in range(1, 10)]
            else:
                thresholds = [(unique_vals[k] + unique_vals[k+1])/2.0 for k in range(len(unique_vals)-1)]
                
            for thresh in thresholds:
                left_y = [y[i] for i in range(n_samples) if X[i][idx] <= thresh]
                right_y = [y[i] for i in range(n_samples) if X[i][idx] > thresh]
                if not left_y or not right_y: continue
                sse = self._calculate_sse(left_y) + self._calculate_sse(right_y)
                if sse < best_sse:
                    best_sse, best_idx, best_thresh = sse, idx, thresh
        return best_idx, best_thresh

    def _build_tree(self, X, y, depth=0):
        n_samples = len(X)
        n_features = len(X[0])
        if depth >= self.max_depth or n_samples < self.min_samples_split or len(set(y)) == 1:
            return Node(value=sum(y)/len(y) if y else 0.0)
            
        feature_indices = random.sample(range(n_features), min(n_features, self.max_features)) if self.max_features else list(range(n_features))
        best_idx, best_thresh = self._best_split(X, y, feature_indices)
        if best_idx is None: return Node(value=sum(y)/len(y) if y else 0.0)
        
        left_idx = [i for i in range(n_samples) if X[i][best_idx] <= best_thresh]
        right_idx = [i for i in range(n_samples) if X[i][best_idx] > best_thresh]
        
        left = self._build_tree([X[i] for i in left_idx], [y[i] for i in left_idx], depth + 1)
        right = self._build_tree([X[i] for i in right_idx], [y[i] for i in right_idx], depth + 1)
        return Node(feature=best_idx, threshold=best_thresh, left=left, right=right)

    def predict(self, X):
        return [self._predict_row(self.root, row) for row in X]

    def _predict_row(self, node, row):
        if node.is_leaf(): return node.value
        if row[node.feature] <= node.threshold: return self._predict_row(node.left, row)
        return self._predict_row(node.right, row)

class RandomForestRegressorScratch:
    """Random Forest Regressor"""
    def __init__(self, n_estimators=30, max_depth=6, min_samples_split=5, max_features=None, random_state=42):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.random_state = random_state
        self.trees = []

    def fit(self, X, y):
        random.seed(self.random_state)
        n_samples = len(X)
        n_features = len(X[0])
        if self.max_features is None: self.max_features = max(1, n_features // 3)
        
        self.trees = []
        for i in range(self.n_estimators):
            indices = [random.randint(0, n_samples - 1) for _ in range(n_samples)]
            bootstrap_X = [X[idx] for idx in indices]
            bootstrap_y = [y[idx] for idx in indices]
            
            tree = DecisionTreeRegressorScratch(max_depth=self.max_depth, min_samples_split=self.min_samples_split, max_features=self.max_features)
            tree.fit(bootstrap_X, bootstrap_y)
            self.trees.append(tree)

    def predict(self, X):
        tree_preds = [tree.predict(X) for tree in self.trees]
        return [sum(tree_preds[t][i] for t in range(self.n_estimators))/self.n_estimators for i in range(len(X))]

def get_metrics(y_true, y_pred):
    n = len(y_true)
    mae = sum(abs(y_true[i] - y_pred[i]) for i in range(n)) / n
    mse = sum((y_true[i] - y_pred[i])**2 for i in range(n)) / n
    rmse = math.sqrt(mse)
    mean_y = sum(y_true) / n
    ss_tot = sum((y_true[i] - mean_y)**2 for i in range(n))
    ss_res = sum((y_true[i] - y_pred[i])**2 for i in range(n))
    r2 = 1.0 - (ss_res / ss_tot) if ss_tot > 0 else 0.0
    return {"MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2}

## 3. Análise Exploratória (EDA)

Abaixo importamos as bibliotecas de plotagem e exibimos a análise gráfica.

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Carregar os dados originais no pandas para EDA
df_mat = pd.read_csv("data/student-mat.csv", sep=";")
df_por = pd.read_csv("data/student-por.csv", sep=";")

# Plotar distribuição de G3
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df_mat['G3'], bins=20, kde=True, color='#1a2b4c', ax=axes[0])
axes[0].set_title("Distribuição de Notas Finais (G3) - Matemática")
axes[0].set_xlabel("Nota G3")
axes[0].set_ylabel("Contagem")

sns.histplot(df_por['G3'], bins=20, kde=True, color='#cda250', ax=axes[1])
axes[1].set_title("Distribuição de Notas Finais (G3) - Português")
axes[1].set_xlabel("Nota G3")
axes[1].set_ylabel("Contagem")
plt.show()

ImportError: DLL load failed while importing _generator: Uma política de Controle de Aplicativo bloqueou este arquivo.

In [ ]:
# Mostrar correlação das variáveis numéricas originais
num_cols = ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 
            'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3']

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df_mat[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
ax.set_title("Matriz de Correlação das Variáveis Numéricas (Matemática)")
plt.show()

## 4. Pré-processamento e Modelagem

Abaixo processamos os dados utilizando as nossas classes manuais e ajustamos os três modelos regressivos.

In [ ]:
# 1. Carregar usando nossa lógica de extração do zero
from ml_from_scratch import load_and_preprocess_scratch

# Matemática
X, y, feat_names, num_names, cat_names = load_and_preprocess_scratch("data/student-mat.csv")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Padronizar numéricos
num_indices = list(range(len(num_names)))
scaler = StandardScalerScratch()
X_train_num = [[row[idx] for idx in num_indices] for row in X_train]
scaler.fit(X_train_num)

# Substituir em treino e teste
X_train_scaled = [scaler.transform([X_train[i][:len(num_names)]])[0] + X_train[i][len(num_names):] for i in range(len(X_train))]
X_test_scaled = [scaler.transform([X_test[i][:len(num_names)]])[0] + X_test[i][len(num_names):] for i in range(len(X_test))]

# 2. Treinar os modelos
print("Treinando Regressão Linear...")
lr = LinearRegressionGD(lr=0.05, epochs=1500)
lr.fit(X_train_scaled, y_train)
pred_lr = lr.predict(X_test_scaled)

print("Treinando Árvore de Decisão...")
dt = DecisionTreeRegressorScratch(max_depth=5, min_samples_split=5)
dt.fit(X_train_scaled, y_train)
pred_dt = dt.predict(X_test_scaled)

print("Treinando Random Forest...")
rf = RandomForestRegressorScratch(n_estimators=30, max_depth=6, min_samples_split=5, random_state=42)
rf.fit(X_train_scaled, y_train)
pred_rf = rf.predict(X_test_scaled)

# Exibir resultados de teste
print("\n--- RESULTADOS MATEMÁTICA ---")
for name, preds in [("Regressão Linear", pred_lr), ("Árvore de Decisão", pred_dt), ("Random Forest", pred_rf)]:
    met = get_metrics(y_test, preds)
    print(f"{name}: MAE: {met['MAE']:.3f} | MSE: {met['MSE']:.3f} | RMSE: {met['RMSE']:.3f} | R²: {met['R2']:.3f}")

## 5. Visualizações de Avaliação

Vamos plotar o gráfico comparativo de Real vs Previsto e Resíduos dos modelos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Real vs Previsto
axes[0].scatter(y_test, pred_lr, color='#1a2b4c', alpha=0.7, label='Regr. Linear')
axes[0].scatter(y_test, pred_rf, color='#cda250', alpha=0.7, label='Random Forest')
axes[0].plot([0, 20], [0, 20], 'r--', lw=2)
axes[0].set_title("Real vs. Previsto (Teste - Matemática)")
axes[0].set_xlabel("Nota Real")
axes[0].set_ylabel("Nota Prevista")
axes[0].legend()

# Gráfico de Resíduos
res_lr = np.array(y_test) - np.array(pred_lr)
res_rf = np.array(y_test) - np.array(pred_rf)
axes[1].scatter(pred_lr, res_lr, color='#1a2b4c', alpha=0.7, label='Regr. Linear')
axes[1].scatter(pred_rf, res_rf, color='#cda250', alpha=0.7, label='Random Forest')
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_title("Gráfico de Resíduos")
axes[1].set_xlabel("Valores Previstos")
axes[1].set_ylabel("Resíduo (Real - Previsto)")
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Discussão e Conclusão

Analisando os resultados de teste para Matemática:
1. Os modelos atingem R² satisfatório (acima de 70%), com forte dependência de G1 e G2.
2. A regressão de Português (no script `run_analysis_scratch.py`) atinge coeficientes de R² ainda mais elevados (cerca de 85%).
3. O absenteísmo (`absences`) e o histórico de reprovações (`failures`) provaram ser os fatores demográficos extracurriculares de maior impacto negativo na nota final.